In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

# 显示所有列
pd.set_option('display.max_columns', None)

print("库导入成功！")

库导入成功！


In [3]:
# -----------------------------------------读取 CSV文件---------------------------------------------------
df = pd.read_csv(r"F:\正大杯\code\data_raw(code).csv", encoding='gbk')

# 查看数据基本信息
print(f"数据形状：{df.shape}")
print(f"\n列名：\n{df.columns.tolist()}")
print(f"\n数据类型：\n{df.dtypes.value_counts()}")

数据形状：(400, 42)

列名：
['age_group', 'has_bought_clothes', 'monthly_expense', 'info_channels', 'design_importance', 'texture_importance', 'price_importance', 'limited_importance', 'designer_importance', 'planned_buy', 'impulse_buy', 'collect_series', 'follow_trends', 'secondhand_enjoy', 'browsing_satisfaction', 'unboxing_satisfaction', 'dressing_satisfaction', 'photo_satisfaction', 'organizing_satisfaction', 'stress_relief', 'aesthetic_achievement', 'companionship_warmth', 'control_safety', 'escape_reality', 'emotion_regulation', 'style_represents_me', 'expression_of_attitude', 'reflects_personality', 'collection_reflects_values', 'ideal_self_carrier', 'belonging_via_style', 'sense_of_belonging_communication', 'recognition_from_knowledge', 'proud_of_collection', 'identity_label', 'community_participation', 'relationship_metaphor', 'shopping_experience_text', 'gender', 'occupation_status', 'city_level', 'monthly_income']

数据类型：
object    42
Name: count, dtype: int64


In [4]:
# -------------------------------------------数据清洗-----------------------------------------
# 1. 检查缺失值
print("缺失值统计：")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. 检查重复值
print(f"\n重复行数：{df.duplicated().sum()}")

# 3. 过滤无效样本（只保留购买了娃衣的用户）
# 数据应该已经都是"是"
print(f"\n是否购买娃衣：\n{df['has_bought_clothes'].value_counts()}")

# 4. 删除不需要的列（如果有）
# df = df.drop(columns=['不需要的列名'])

缺失值统计：
Series([], dtype: int64)

重复行数：0

是否购买娃衣：
has_bought_clothes
是    400
Name: count, dtype: int64


In [5]:
# 定义映射字典
likert_map = {
    '非常不同意': 1,
    '不同意': 2,
    '一般': 3,
    '同意': 4,
    '非常同意': 5
}

satisfaction_map = {
    '毫无满足': 1,
    '较少满足': 2,
    '一般': 3,
    '比较满足': 4,
    '极大满足': 5
}

frequency_map = {
    '从未获得': 1,
    '很少获得': 2,
    '有时获得': 3,
    '经常获得': 4,
    '总是获得': 5
}

# 需要映射的列
likert_cols = [
    'design_importance', 'texture_importance', 'price_importance', 
    'limited_importance', 'designer_importance', 'planned_buy', 
    'impulse_buy', 'collect_series', 'follow_trends', 'secondhand_enjoy',
    'style_represents_me', 'expression_of_attitude', 'reflects_personality',
    'collection_reflects_values', 'ideal_self_carrier', 'belonging_via_style',
    'sense_of_belonging_communication', 'recognition_from_knowledge',
    'proud_of_collection', 'identity_label'
]

satisfaction_cols = [
    'browsing_satisfaction', 'unboxing_satisfaction', 
    'dressing_satisfaction', 'photo_satisfaction', 'organizing_satisfaction'
]

frequency_cols = [
    'stress_relief', 'aesthetic_achievement', 'companionship_warmth',
    'control_safety', 'escape_reality'
]

# 执行映射
for col in likert_cols:
    if col in df.columns:
        df[col + '_score'] = df[col].map(likert_map)

for col in satisfaction_cols:
    if col in df.columns:
        df[col + '_score'] = df[col].map(satisfaction_map)

for col in frequency_cols:
    if col in df.columns:
        df[col + '_score'] = df[col].map(frequency_map)

# 检查映射是否成功
print("映射后的数值列示例：")
print(df[['impulse_buy', 'impulse_buy_score']].head())

映射后的数值列示例：
  impulse_buy  impulse_buy_score
0          一般                  3
1          同意                  4
2          一般                  3
3          一般                  3
4          同意                  4


In [36]:
# 情感补偿得分（取相关维度的平均）
# 情感补偿得分 = 压力释放与情绪放松 , 情感陪伴的温暖感 ,  暂时逃离现实的抽离感
emotion_cols = ['stress_relief_score', 'companionship_warmth_score', 'escape_reality_score']
df['emotion_compensation_score'] = df[emotion_cols].mean(axis=1)

# 身份认同得分（自我表达 + 群体归属）
# 自我表达 = 个人的审美品味 , 表达我的个性与态度 , 承载着我理想自我或渴望的形象
self_expression_cols = ['style_represents_me_score', 'expression_of_attitude_score', 'ideal_self_carrier_score']
# 群体归属 = 感到属于某个群体 , 社群中交流娃衣获得了归属感 , “娃衣爱好者”是重要的身份标签之一
group_belonging_cols = ['belonging_via_style_score', 'sense_of_belonging_communication_score', 'identity_label_score']
df['self_expression_score'] = df[self_expression_cols].mean(axis=1)
df['group_belonging_score'] = df[group_belonging_cols].mean(axis=1)
df['identity_score'] = df[['self_expression_score', 'group_belonging_score']].mean(axis=1)

# 消费金额数值化（用于后续分析）
expense_map = {
    'A. 100元以下': 1,
    'B. 101-300元': 2,
    'C. 301-500元': 3,
    'D. 501-1000元': 4,
    'E. 1000元以上': 5
}
df['monthly_expense_score'] = df['monthly_expense'].map(expense_map)

print("核心指标构建完成！")
print(f"情感补偿得分范围：{df['emotion_compensation_score'].min()} - {df['emotion_compensation_score'].max()}")
print(f"身份认同得分范围：{df['identity_score'].min()} - {df['identity_score'].max()}")

核心指标构建完成！
情感补偿得分范围：2.6666666666666665 - 5.0
身份认同得分范围：2.6666666666666665 - 4.833333333333334


In [28]:
# 保存为新的 CSV，供后续分析使用
df.to_csv(r'F:\正大杯\code\data_cleaned.csv', index=False, encoding='gbk')
print("清洗后的数据已保存到 data_cleaned.csv")

清洗后的数据已保存到 data_cleaned.csv
